# 🧪 Taller - Gestos con Cámara Web: Control Visual con MediaPipe

Para este notebook específico, se quiso desarrollar un Pong controlado por gestos de la mano.

### Importar librerias

In [17]:
import cv2
import mediapipe as mp
import numpy as np
import time
import random

## Metadatos de juego

In [18]:
# --- Parámetros del juego ---
WIDTH, HEIGHT = 640, 480
PADDLE_WIDTH, PADDLE_HEIGHT = 10, 80
BALL_RADIUS = 10
BALL_SPEED_X, BALL_SPEED_Y = 5, 5

# --- Estado del juego ---
player_score = 0
cpu_score = 0
ball_x, ball_y = WIDTH // 2, HEIGHT // 2
ball_speed_x, ball_speed_y = BALL_SPEED_X, BALL_SPEED_Y
player_paddle_y = HEIGHT // 2 - PADDLE_HEIGHT // 2
cpu_paddle_y = HEIGHT // 2 - PADDLE_HEIGHT // 2

# --- Colores ---
dark_theme = {
    'bg': (0, 0, 0),
    'fg': (255, 255, 255)
}
light_theme = {
    'bg': (255, 255, 255),
    'fg': (0, 0, 0)
}
theme = dark_theme

# --- Temporizadores ---
last_color_change_time = 0
last_mirror_toggle_time = 0

mirror_mode = False

## Detección de gestos

In [19]:
# --- Control por gestos ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

# --- Funciones auxiliares ---
def is_finger_extended(hand, tip_id, pip_id):
    return hand.landmark[tip_id].y < hand.landmark[pip_id].y

def hand_open(hand):
    return all([
        is_finger_extended(hand, mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        is_finger_extended(hand, mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        is_finger_extended(hand, mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        is_finger_extended(hand, mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP),
    ])

def hand_closed(hand):
    return all([
        not is_finger_extended(hand, mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        not is_finger_extended(hand, mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        not is_finger_extended(hand, mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        not is_finger_extended(hand, mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP),
    ])

## Lógica de juego

In [20]:
def move_ball():
    global ball_x, ball_y, ball_speed_x, ball_speed_y, player_score, cpu_score

    ball_x += ball_speed_x
    ball_y += ball_speed_y

    if ball_y - BALL_RADIUS < 0 or ball_y + BALL_RADIUS > HEIGHT:
        ball_speed_y *= -1

    if ball_x - BALL_RADIUS < PADDLE_WIDTH and player_paddle_y < ball_y < player_paddle_y + PADDLE_HEIGHT:
        ball_speed_x *= -1
    elif ball_x + BALL_RADIUS > WIDTH - PADDLE_WIDTH and cpu_paddle_y < ball_y < cpu_paddle_y + PADDLE_HEIGHT:
        ball_speed_x *= -1

    if ball_x < 0:
        cpu_score += 1
        reset_ball()
    if ball_x > WIDTH:
        player_score += 1
        reset_ball()

def reset_ball():
    global ball_x, ball_y, ball_speed_x, ball_speed_y
    ball_x, ball_y = WIDTH // 2, HEIGHT // 2
    ball_speed_x *= random.choice([-1, 1])
    ball_speed_y *= random.choice([-1, 1])

In [21]:
def draw(frame):
    frame[:] = theme['bg']

    # Paletas
    cv2.rectangle(frame, (0, int(player_paddle_y)),
                  (PADDLE_WIDTH, int(player_paddle_y + PADDLE_HEIGHT)), theme['fg'], -1)
    cv2.rectangle(frame, (WIDTH - PADDLE_WIDTH, int(cpu_paddle_y)),
                  (WIDTH, int(cpu_paddle_y + PADDLE_HEIGHT)), theme['fg'], -1)

    # Pelota
    cv2.circle(frame, (int(ball_x), int(ball_y)), BALL_RADIUS, theme['fg'], -1)

    # Puntuación
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(frame, f'Jugador: {player_score}', (50, 30), font, 0.8, theme['fg'], 2)
    cv2.putText(frame, f'CPU: {cpu_score}', (WIDTH - 180, 30), font, 0.8, theme['fg'], 2)

In [22]:
# --- Main loop ---
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    gesture = "Ninguno"

    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        wrist_y = hand.landmark[mp_hands.HandLandmark.WRIST].y
        player_paddle_y = int(wrist_y * HEIGHT) - PADDLE_HEIGHT // 2
        player_paddle_y = np.clip(player_paddle_y, 0, HEIGHT - PADDLE_HEIGHT)

        current_time = time.time()

        if current_time - last_color_change_time > 1:
            if hand_open(hand):
                theme = light_theme if theme == dark_theme else dark_theme
                last_color_change_time = current_time
                gesture = "Abierta"

        if current_time - last_mirror_toggle_time > 1:
            if hand_closed(hand):
                mirror_mode = not mirror_mode
                last_mirror_toggle_time = current_time
                gesture = "Cerrada"

        if gesture == "Ninguno":
            if hand_open(hand):
                gesture = "Abierta"
            elif hand_closed(hand):
                gesture = "Cerrada"

        mp_draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

    # Movimiento CPU
    cpu_paddle_y += (ball_y - (cpu_paddle_y + PADDLE_HEIGHT // 2)) * 0.05
    cpu_paddle_y = np.clip(cpu_paddle_y, 0, HEIGHT - PADDLE_HEIGHT)

    move_ball()

    # --- Dibujar juego ---
    game_frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    draw(game_frame)
    if mirror_mode:
        game_frame = cv2.flip(game_frame, 1)

    # Mostrar cámaras
    cv2.putText(frame, f"Gesto: {gesture}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("Pong con gestos", game_frame)
    cv2.imshow("Camara + Mano", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()